# ภาษีที่ชื่อ Volatility Drag — Leveraged ETF Lab
### 0DTE Labs · เครื่องมือประกอบบทความ "ทำไม 3x ETF ถึงไม่ให้กำไร 3 เท่า"

Notebook นี้สาธิต **volatility drag** ใน leveraged ETF (LETF) ด้วยการ simulate จริง
ตรวจสอบสูตร **Avellaneda-Zhang (2010)** และเทียบกับ **ข้อมูล ETF จริง** (SPY/SSO/UPRO และ QQQ/QLD/TQQQ)
กราฟทั้งหมดใช้ **Plotly** (interactive — ซูม/hover ได้)

> เนื้อหาเชิงการศึกษา ไม่ใช่คำแนะนำการลงทุน · โมเดลจำลองใช้ GBM รายวันซึ่งลดทอนความจริง

**สารบัญ**
1. Intuition - ตัวอย่าง 2 วันที่ดัชนีเสมอตัวแต่ 3x ขาดทุน
2. Simulate path เดียว (Plotly)
3. ตรวจสอบสูตร Avellaneda-Zhang เชิงประจักษ์
4. Drag เป็นฟังก์ชันของ volatility และ leverage
5. ข้อมูล ETF จริง ด้วย yfinance (รันเซลล์นี้บนเครื่องที่มีเน็ต)


In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(7)
COL = {1:'#9aa4b2', 2:'#4a9eff', 3:'#ff7a1a'}   # สี 1x / 2x / 3x (ธีม 0DTE Labs)
TEMPLATE = 'plotly_dark'

## 1. Intuition ก่อน - ดัชนีกลับที่เดิม แต่ 3x ขาดทุน

สมมติแค่ 2 วัน: วันแรกดัชนี **+10%** วันที่สอง **-9.09%** -> กลับมาที่เดิมพอดี (1.10 x 0.9091 = 1.00)

In [2]:
r = [0.10, -1/11]
idx = np.prod([1+x for x in r])
x3  = np.prod([1+3*x for x in r])
print(f"ดัชนี (1x): {(idx-1)*100:+.2f}%   -> เสมอตัว")
print(f"3x ETF   : {(x3-1)*100:+.2f}%   <- ขาดทุน ทั้งที่ underlying ไม่ไปไหน")

ดัชนี (1x): +0.00%   -> เสมอตัว
3x ETF   : -5.45%   <- ขาดทุน ทั้งที่ underlying ไม่ไปไหน


3x วันแรก +30% (เป็น 1.30) วันที่สอง -27.3% (1.30 x 0.727 = 0.945) -> **ขาดทุน 5.5%**
ความไม่สมมาตรของการทบต้นคือรากของ drag ยิ่ง leverage สูง ช่องว่างยิ่งถ่าง

## 2. Simulate path เดียว (Plotly)

จำลองผลตอบแทน**รายวัน**ของดัชนีแบบ GBM แล้วสร้าง LETF ที่ให้ beta เท่าของผลตอบแทนรายวัน
(reset ทุกวัน = หัวใจของ path-dependence)

In [3]:
def simulate(mu, sigma, days, betas=(1,2,3), fee=0.0, rng=None):
    rng = rng or np.random.default_rng()
    dt = 1/252
    r = mu*dt + sigma*np.sqrt(dt)*rng.standard_normal(days)
    daily_fee = fee*dt
    paths = {b: np.concatenate([[1.0], np.cumprod(1 + b*r - daily_fee)]) for b in betas}
    rv = np.sum(r**2)
    return paths, r, rv

for s in range(1, 500):
    rng = np.random.default_rng(s)
    paths, r, rv = simulate(mu=0.0, sigma=0.40, days=252, fee=0.009, rng=rng)
    if abs(paths[1][-1]-1) < 0.02:
        break

fig = go.Figure()
for b, p in paths.items():
    fig.add_trace(go.Scatter(y=p*100, mode='lines', name=f'{b}x',
                             line=dict(color=COL[b], width=3 if b==3 else 2)))
fig.add_hline(y=100, line=dict(color='gray', dash='dash'))
fig.update_layout(template=TEMPLATE, title='Sideways (mu=0%, sigma=40%, 1 year) - start at 100',
                  xaxis_title='trading day', yaxis_title='portfolio value', height=430)
fig.show()
for b, p in paths.items():
    print(f"{b}x: actual return {(p[-1]-1)*100:+.1f}%")

1x: actual return +0.7%
2x: actual return -12.4%
3x: actual return -35.0%


ดัชนีแทบไม่ขยับ แต่ 3x ติดลบชัดเจน - drag กัดกินโดยไม่มีใครเก็บค่าธรรมเนียมพิเศษ

## 3. ตรวจสอบสูตร Avellaneda-Zhang (2010)

L_T / L_0 ≈ (S_T / S_0)^beta · exp[ -0.5 · beta(beta-1) · sum(r_t^2) ]

- (S_T/S_0)^beta = ส่วน leverage ทบต้น ถ้าโลกไม่มีความผันผวน
- exp[-0.5·beta(beta-1)·RV] = drag ที่หดมูลค่าลงตาม realized variance

ทดสอบว่าสูตรทำนายค่า LETF จริงได้แม่นแค่ไหน บน 2,000 path สุ่ม

In [4]:
rng = np.random.default_rng(2024)
beta = 3
actual, predicted = [], []
for _ in range(2000):
    sigma = rng.uniform(0.10, 0.80)
    mu    = rng.uniform(-0.15, 0.25)
    days  = int(rng.integers(40, 400))
    paths, r, rv = simulate(mu, sigma, days, betas=(1, beta), fee=0.0, rng=rng)
    S, L = paths[1][-1], paths[beta][-1]
    predicted.append((S**beta)*np.exp(-0.5*beta*(beta-1)*rv))
    actual.append(L)

actual, predicted = np.array(actual), np.array(predicted)
err = np.abs(actual-predicted)/actual
print(f"median relative error = {np.median(err)*100:.3f}%")
print(f"95th pct error        = {np.percentile(err,95)*100:.3f}%")

lim = [min(actual.min(),predicted.min()), max(actual.max(),predicted.max())]
fig = go.Figure()
fig.add_trace(go.Scatter(x=predicted, y=actual, mode='markers',
                         marker=dict(size=4, color='#ff7a1a', opacity=0.35), name='paths'))
fig.add_trace(go.Scatter(x=lim, y=lim, mode='lines', line=dict(color='white', dash='dash'), name='y = x'))
fig.update_layout(template=TEMPLATE, title='3x ETF: Avellaneda-Zhang formula vs simulation (2000 paths)',
                  xaxis_title='formula  S^b * exp(-0.5*b(b-1)*RV)', yaxis_title='actual LETF from sim',
                  height=470)
fig.show()

median relative error = 0.635%
95th pct error        = 12.077%


จุดเกาะเส้น y = x แน่น (error กลางราว 0.6%) -> สูตรจับ drag ได้จริง
ที่เหลือคือ discretization ของโมเดลรายวัน

## 4. Drag โตตาม sigma^2 และ beta(beta-1)

พจน์ drag = 0.5·beta(beta-1)·sigma^2·T ทำนายว่า drag โตตามกำลังสองของ sigma
และแรงขึ้นตาม beta(beta-1): 2x ให้ค่า 1, 3x ให้ค่า 3 (แรงกว่า 3 เท่า ไม่ใช่ 1.5 เท่า)

In [5]:
def mean_drag(beta, sigma, days=252, n=3000):
    rng = np.random.default_rng(int(sigma*1000)+beta)
    gaps = []
    for _ in range(n):
        paths, r, rv = simulate(0.0, sigma, days, betas=(1, beta), fee=0.0, rng=rng)
        gaps.append((paths[1][-1]**beta) - paths[beta][-1])
    return np.mean(gaps)

sigmas = np.linspace(0.10, 0.80, 8)
fig = go.Figure()
for beta in (2, 3):
    sim_drag  = [mean_drag(beta, s)*100 for s in sigmas]
    theo_drag = [(1-np.exp(-0.5*beta*(beta-1)*s**2))*100 for s in sigmas]
    fig.add_trace(go.Scatter(x=sigmas*100, y=sim_drag, mode='lines+markers',
                             line=dict(color=COL[beta]), name=f'{beta}x (simulation)'))
    fig.add_trace(go.Scatter(x=sigmas*100, y=theo_drag, mode='lines',
                             line=dict(color=COL[beta], dash='dash'), name=f'{beta}x (formula)'))
fig.update_layout(template=TEMPLATE, title='Drag grows with sigma^2 and beta(beta-1)',
                  xaxis_title='annual volatility sigma (%)', yaxis_title='mean drag (% of base, mu=0)',
                  height=450)
fig.show()
print(f"drag term ratio  3x : 2x = {(0.5*3*2)/(0.5*2*1):.0f} : 1  (not 1.5 : 1)")

drag term ratio  3x : 2x = 3 : 1  (not 1.5 : 1)


## 5. ข้อมูล ETF จริง (yfinance)

> รันเซลล์นี้บนเครื่องที่ต่อเน็ต (ติดตั้งก่อน: pip install yfinance)
> ถ้าเน็ตถูกบล็อก เซลล์จะข้ามอย่างนุ่มนวลและพิมพ์ข้อความบอก

เทียบราคาจริงของสองตระกูล LETF:
- S&P 500: SPY (1x) · SSO (2x) · UPRO (3x)
- Nasdaq-100: QQQ (1x) · QLD (2x) · TQQQ (3x)

แล้วเอา realized variance จากผลตอบแทนจริงของ 1x ไปใส่สูตร Avellaneda-Zhang ทำนาย 3x เทียบของจริง

In [6]:
START = '2015-01-01'
FAMILIES = {'S&P 500': ['SPY','SSO','UPRO'], 'Nasdaq-100': ['QQQ','QLD','TQQQ']}

prices = {}
HAVE_DATA = False
try:
    import yfinance as yf, pandas as pd
    tickers = sum(FAMILIES.values(), [])
    raw = yf.download(tickers, start=START, auto_adjust=True, progress=False)
    cols = getattr(raw, 'columns', None)
    if cols is not None and hasattr(cols, 'levels') and 'Close' in cols.levels[0]:
        close = raw['Close']
    else:
        close = raw
    close = close.dropna()
    assert not close.empty, 'empty frame'
    prices = {t: close[t] for t in tickers}
    HAVE_DATA = True
    print(f'ดึงข้อมูลสำเร็จ: {close.index[0].date()} -> {close.index[-1].date()}  ({len(close)} วัน)')
except Exception as e:
    print('ดึงข้อมูลจริงไม่ได้ (ต้องมีเน็ต + pip install yfinance).')
    print('รายละเอียด:', repr(e))

ดึงข้อมูลสำเร็จ: 2015-01-02 -> 2026-07-07  (2893 วัน)


In [7]:
if HAVE_DATA:
    import pandas as pd
    fig = make_subplots(rows=1, cols=2, subplot_titles=list(FAMILIES.keys()))
    for col, (fam, ts) in enumerate(FAMILIES.items(), start=1):
        base = pd.concat([prices[t] for t in ts], axis=1).dropna()
        base.columns = ts
        norm = base / base.iloc[0] * 100
        for b, t in zip((1,2,3), ts):
            fig.add_trace(go.Scatter(x=norm.index, y=norm[t], name=f'{t} ({b}x)',
                          line=dict(color=COL[b], width=2.5 if b==3 else 1.8)), row=1, col=col)
    fig.update_layout(template=TEMPLATE, height=470,
                      title=f'growth of $100 since {START} (real data, auto-adjusted)')
    fig.show()

    print(f"{'Family':<12}{'ETF':<7}{'actual':>10}{'naive b*x':>11}{'drag':>10}")
    print('-'*50)
    for fam, ts in FAMILIES.items():
        base = pd.concat([prices[t] for t in ts], axis=1).dropna(); base.columns = ts
        rets = base.pct_change().dropna()
        tot = base.iloc[-1]/base.iloc[0] - 1
        idx_ret = tot[ts[0]]
        for b, t in zip((1,2,3), ts):
            if b == 1:
                print(f"{fam:<12}{t:<7}{tot[t]*100:>9.0f}%{'':>11}{'':>10}")
            else:
                naive = b*idx_ret; drag = tot[t]-naive
                print(f"{'':<12}{t:<7}{tot[t]*100:>9.0f}%{naive*100:>10.0f}%{drag*100:>9.0f}%")
        rv_1x = np.sum(rets[ts[0]].values**2)
        S = base[ts[0]].iloc[-1]/base[ts[0]].iloc[0]
        pred_3x = S**3 * np.exp(-0.5*3*2*rv_1x)
        act_3x  = base[ts[2]].iloc[-1]/base[ts[2]].iloc[0]
        print(f"   -> {fam}: formula 3x = {(pred_3x-1)*100:+.0f}%  vs actual = {(act_3x-1)*100:+.0f}%  (RV={rv_1x:.3f})")
        print()
else:
    print('ข้ามส่วนข้อมูลจริง (ไม่มีข้อมูล)')

Family      ETF        actual  naive b*x      drag
--------------------------------------------------
S&P 500     SPY          341%                     
            SSO          794%       681%      113%
            UPRO        1266%      1022%      244%
   -> S&P 500: formula 3x = +2822%  vs actual = +1266%  (RV=0.358)

Nasdaq-100  QQQ          650%                     
            QLD         2044%      1300%      744%
            TQQQ        3673%      1951%     1722%
   -> Nasdaq-100: formula 3x = +7893%  vs actual = +3673%  (RV=0.555)



**อ่านผลอย่างระวัง:** ในช่วง bull ยาว (เช่นตั้งแต่ 2015) ตลาด trending พอที่ compounding
มักช่วยให้ 3x จบสูงกว่า naive 3x หรือใกล้เคียง — drag ไม่ได้แพ้เสมอ ขึ้นกับ path
ลองเปลี่ยน START เป็น '2022-01-01' จะเห็นปีที่ผันผวนออกข้าง drag กัดกิน 3x ชัดเจน
(สูตร Avellaneda-Zhang ทำนาย 3x จาก RV ของ 1x ได้ใกล้ของจริง = ยืนยัน drag บนข้อมูลจริง)

## สรุป

1. Volatility drag เป็นเลขคณิตทบต้น ไม่ใช่ค่าธรรมเนียม - เกิดจากการ reset leverage รายวัน
2. สูตร Avellaneda-Zhang แม่น ทั้งบน simulation และข้อมูล ETF จริง
3. 3x เจ็บกว่า 2x ราว 3 เท่า (จาก beta(beta-1)) และ drag โตตาม sigma^2
4. Drag ไม่ได้แพ้เสมอ - ตลาด trending แรง compounding อาจกลบ drag; ทั้งหมดขึ้นกับ path

**อ้างอิง**
- Avellaneda, M. & Zhang, S. (2010). Path-Dependence of Leveraged ETF Returns. SIAM J. Financial Math. 1, 586-603.
- Cheng, M. & Madhavan, A. (2009). The Dynamics of Leveraged and Inverse-Exchange Traded Funds. J. Investment Management.
